<a href="https://colab.research.google.com/github/csu-techhub/QOS-Labs/blob/main/Lab3.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 3 — QAOA: Ideal Simulator, Noise, and Real Hardware
**Quantum Optimization and Simulation — QAOA Laboratory Series**

> Qiskit displays measured bitstrings in the order `q_(n-1)...q_0`. When we discuss graph nodes, this notebook often converts them to `q_0...q_(n-1)` using `q0_first(...)`.

## Learning goals
- Compare an ideal Aer simulation with a noisy simulation.
- Transpile the same QAOA circuit for an IBM backend.
- Optionally submit a real-hardware job.
- Relate circuit depth/two-qubit gates to NISQ performance.

**Important:** The real-hardware section is optional if you do not have IBM Quantum access or a queue is long.

In [ ]:
# Run this once at the beginning of a fresh Google Colab session.
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-optimization~=0.7" "qiskit-ibm-runtime~=0.46"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_aer.primitives import SamplerV2

SEED = 123
SHOTS = 2048
sampler = SamplerV2(default_shots=SHOTS, seed=SEED)

def run_counts(qc, shots=SHOTS):
    "Run a measured circuit with Aer SamplerV2 and return counts."
    result = sampler.run([qc], shots=shots).result()
    return result[0].data.meas.get_counts()

def q0_first(qiskit_bitstring):
    "Convert Qiskit's displayed q_(n-1)...q_0 bitstring to q_0...q_(n-1)."
    return qiskit_bitstring.replace(" ", "")[::-1]

## run_counts() for noisy model (FakeSherbrooke)

In [ ]:
# noisy fake backend ──────────
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit import transpile

fake_backend  = FakeSherbrooke()
noisy_backend = AerSimulator.from_backend(fake_backend)

noisy_pm      = generate_preset_pass_manager(backend=noisy_backend,
                                             optimization_level=1,
                                             seed_transpiler=SEED)
noisy_sampler = SamplerV2.from_backend(noisy_backend, seed=SEED)

def run_counts_noisy(qc, shots=SHOTS):
    "Transpile to the fake backend's ISA, run with its noise model, return counts."
    isa_qc = noisy_pm.run(qc)
    result = noisy_sampler.run([isa_qc], shots=shots).result()
    return result[0].data.meas.get_counts()

In [ ]:
edges5 = [(0,3),(0,4),(1,3),(1,4),(2,3),(2,4)]

def build_qaoa_5(gamma, beta):
    qc = QuantumCircuit(5)
    qc.h(range(5))
    for i, j in edges5:
        qc.cx(i, j)
        qc.rz(-gamma, j)
        qc.cx(i, j)
    for q in range(5):
        qc.rx(2 * beta, q)
    qc.measure_all()
    return qc

## Part A — Ideal Aer versus FakeSherbrooke: one pamameter set ($\gamma, \beta$)

In [ ]:
gamma = 0.68
beta = 0.39
qc = build_qaoa_5(gamma, beta)

counts = run_counts(qc, shots=4096)
noisy_counts = run_counts_noisy(qc, shots=4096)

#plot_histogram(noisy_counts)
plot_histogram([counts, noisy_counts], legend=["Ideal Aer", "Fake ibm_sherbrooke"])

**Expected:** the noisy distribution is typically flatter/spread out compared with the ideal one.

## Part B — Ideal Aer versus FakeSherbrooke: Run COBYLA

In [ ]:
def cut_value_q0first(bitstring, edges):
    """
    Calculate the cut value for a given bitstring graph partitioning.
    """
    total_cut_edges = 0
    for vertex_a, vertex_b in edges:
        if bitstring[vertex_a] != bitstring[vertex_b]:
            total_cut_edges += 1
    return total_cut_edges

def expected_cut_from_counts(counts, edges):
    """
    Calculate the expected (average) cut value from a distribution of measurement outcomes.
    """
    # Compute total number of measurement samples
    total_shots = sum(counts.values())

    # Handle empty input gracefully to avoid DivisionByZeroError
    if total_shots == 0:
        return 0.0

    weighted_cut_sum = 0.0
    for bitstring_key, count in counts.items():
        formatted_bitstring = q0_first(bitstring_key)
        cut_value = cut_value_q0first(formatted_bitstring, edges)
        weighted_cut_sum += cut_value * count

    expected_value = weighted_cut_sum / total_shots
    return expected_value

In [ ]:
from scipy.optimize import minimize

history = []
history_noisy = []

def objective_p1(params):
    gamma, beta = params
    qc = build_qaoa_5(gamma, beta)
    counts = run_counts(qc, shots=1024)
    avg_cut = expected_cut_from_counts(counts, edges5)
    history.append(avg_cut)
    return -avg_cut


def objective_p1_noisy(params):
    gamma, beta = params
    qc = build_qaoa_5(gamma, beta)
    counts = run_counts_noisy(qc, shots=1024)
    avg_cut = expected_cut_from_counts(counts, edges5)
    history_noisy.append(avg_cut)
    return -avg_cut

In [ ]:
history.clear()
initial_point = np.array([0.3, 0.6])

result = minimize(
    objective_p1,
    x0=initial_point,
    method="COBYLA",
    options={"maxiter": 40, "rhobeg": 0.5}
)

In [ ]:
history_noisy.clear()
initial_point = np.array([0.3, 0.6])

result_noisy = minimize(
    objective_p1_noisy,
    x0=initial_point,
    method="COBYLA",
    options={"maxiter": 40, "rhobeg": 0.5}
)

print(f"ideal  : best <cut> = {-result.fun:.3f}  at γ={result.x[0]:.3f}, β={result.x[1]:.3f}  ({len(history)} evals)")
print(f"noisy  : best <cut> = {-result_noisy.fun:.3f}  at γ={result_noisy.x[0]:.3f}, β={result_noisy.x[1]:.3f}  ({len(history_noisy)} evals)")

plt.plot(history,       marker="o", ms=3, label="ideal Aer")
plt.plot(history_noisy, marker="s", ms=3, label="FakeSherbrooke")
plt.xlabel("energy evaluation"); plt.ylabel("expected cut"); plt.legend(); plt.grid(alpha=.3)
plt.show()

## Part C — Optional IBM hardware

Part B's loop made thirty calls. On hardware that is thirty separate queue waits, which on the Open plan can mean days and will exhaust a monthly allocation in one sitting.

You need an IBM Quantum account to obtain an API key and CRN. If you do not have an account, create a free Open Plan account here:
https://www.ibm.com/quantum/blog/open-plan-updates

In [ ]:
# ── Part C: the same circuit, on real IBM hardware ──────────────────────
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as RuntimeSampler
#from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# One-time setup on this machine (then never again):
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<YOUR_API_KEY>",
#     instance="<YOUR_INSTANCE_CRN>",
#     set_as_default=True,
# )

service = QiskitRuntimeService()
real_backend = service.least_busy(operational=True, simulator=False, min_num_qubits=5)

print(f"backend      : {real_backend.name}")
print(f"qubits       : {real_backend.num_qubits}")
print(f"queue depth  : {real_backend.status().pending_jobs} jobs")

In [ ]:
real_pm      = generate_preset_pass_manager(backend=real_backend, optimization_level=3)
real_sampler = RuntimeSampler(mode=real_backend)
real_sampler.options.dynamical_decoupling.enable = True
real_sampler.options.twirling.enable_gates       = True

gamma = 0.68
beta = 0.39

qc = build_qaoa_5(gamma, beta)
real_isa = real_pm.run(qc)

print(f"depth {qc.depth()} -> {real_isa.depth()} after mapping to {real_backend.name}")
print(f"2q gates: {real_isa.count_ops().get('ecr', 0) + real_isa.count_ops().get('cz', 0)}")

job = real_sampler.run([real_isa], shots=4096)
print(f"job id: {job.job_id()}   <-- write this down")

In [ ]:
# Your job make take longer due to other jobs in the queue.
# Run this cell again until the job is "DONE".
print(job.status())        # QUEUED | RUNNING | DONE | ERROR | CANCELLED

In [ ]:
real_counts = job.result()[0].data.meas.get_counts()

real_cut = expected_cut_from_counts(real_counts, edges5)
print(f"ideal           <cut> = {expected_cut_from_counts(counts, edges5):.3f}")
print(f"FakeSherbrooke  <cut> = {expected_cut_from_counts(noisy_counts, edges5):.3f}")
print(f"{real_backend.name:<15} <cut> = {real_cut:.3f}")

plot_histogram([counts, noisy_counts, real_counts], legend=["Ideal Aer", "Fake ibm_sherbrooke", "Real hardwre"])

## Congratulations! You just ran a program on a real quantum hardware!!

## Notes

1. The cost layer contains many CNOTs; two-qubit gates are typically noisier than single-qubit gates, so their errors accumulate.
2. Hardware has limited connectivity and a restricted native gate set. The transpiler may insert routing/SWAP operations and decompose gates.
3. A simulator noise model includes only selected error mechanisms and usually cannot reproduce every drift, crosstalk, calibration change, leakage, or correlated error on a live device.